# 튜닝 결과 리뷰 - SBERT + PLM(klue-roberta)

원격에서 만든 `tune_sbert_results.json`, `tune_klue-roberta_results.json`을
`tuning_results/`로 옮겨온 뒤 이 노트북을 실행한다. 여기서는 새 계산을 하지
않고, 두 파일을 합쳐서 **최종 비교표**(`tuning_comparison_all.csv`)와
**사람이 읽는 요약**(`tuning_summary.txt`)을 만든다.

실행 전: `tuning_results/tune_sbert_results.json`, `tune_klue-roberta_results.json`이
이 프로젝트에 옮겨져 있는지 확인.

In [ ]:
import sys
import json
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import common_tuning as ct
from common_tuning import base_common


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


SBERT_RESULTS_PATH = ct.TUNING_RESULTS_DIR / "tune_sbert_results.json"
PLM_RESULTS_PATH = ct.TUNING_RESULTS_DIR / "tune_klue-roberta_results.json"

sbert_results = load_json(SBERT_RESULTS_PATH) if SBERT_RESULTS_PATH.exists() else []
plm_results = load_json(PLM_RESULTS_PATH) if PLM_RESULTS_PATH.exists() else []

if not sbert_results:
    print(f"[경고] {SBERT_RESULTS_PATH} 없음 - tune_sbert.py 결과를 옮겨왔는지 확인")
if not plm_results:
    print(f"[경고] {PLM_RESULTS_PATH} 없음 - tune_klue_roberta.py 결과를 옮겨왔는지 확인")

print(f"SBERT config {len(sbert_results)}개, PLM config {len(plm_results)}개 로드")

In [ ]:
rows = []

for r in sbert_results:
    rows.append({
        "model": "SBERT(ko-sroberta)",
        "config_id": r.get("config_id"),
        "max_length": r.get("max_length"),
        "pooling": r.get("pooling"),
        "lr": r.get("lr"),
        "head": r.get("head"),
        "val_macro_f1": r.get("val_macro_f1"),
        "f1_오심": r.get("f1_오심"),
        "train_time_s": r.get("train_time_s"),
    })

for r in plm_results:
    rows.append({
        "model": "PLM(klue-roberta)",
        "config_id": r.get("config_id"),
        "max_length": r.get("max_length"),
        "pooling": None,
        "lr": r.get("lr"),
        "head": None,
        "val_macro_f1": r.get("val_macro_f1"),
        "f1_오심": r.get("f1_오심"),
        "train_time_s": r.get("train_time_s"),
    })

comparison_df = pd.DataFrame(rows)
comparison_df = comparison_df.sort_values(["model", "val_macro_f1"], ascending=[True, False]).reset_index(drop=True)
comparison_df

In [ ]:
def add_gap_vs_baseline(df):
    out = df.copy()
    out["gap_vs_baseline"] = None
    for model in out["model"].unique():
        mask = out["model"] == model
        base_rows = out[mask & (out["config_id"] == "baseline")]
        if base_rows.empty:
            continue
        base_f1 = base_rows.iloc[0]["val_macro_f1"]
        out.loc[mask, "gap_vs_baseline"] = out.loc[mask, "val_macro_f1"] - base_f1
    return out


comparison_df = add_gap_vs_baseline(comparison_df)
comparison_df

In [ ]:
def summarize_model(model_label, results, varied_keys):
    lines = []
    if not results:
        lines.append(f"[경고] {model_label} 결과 없음")
        return lines

    baseline = next((r for r in results if r["config_id"] == "baseline"), None)
    best_combo = next((r for r in results if r["config_id"] == "best_combo"), None)

    lines.append(f"--- {model_label} ---")
    if baseline:
        lines.append(f"  baseline: {{{', '.join(f'{k}={baseline.get(k)}' for k in varied_keys)}}} "
                      f"-> val_macro_f1={baseline.get('val_macro_f1'):.4f}")

    for r in results:
        if r["config_id"] in ("baseline", "best_combo"):
            continue
        gap = (r.get("val_macro_f1") - baseline.get("val_macro_f1")) if baseline else None
        gap_str = f" (baseline 대비 {gap:+.4f})" if gap is not None else ""
        lines.append(f"  {r['config_id']}: {{{', '.join(f'{k}={r.get(k)}' for k in varied_keys)}}} "
                      f"-> val_macro_f1={r.get('val_macro_f1'):.4f}{gap_str}")

    if best_combo:
        gap = (best_combo.get("val_macro_f1") - baseline.get("val_macro_f1")) if baseline else None
        gap_str = f" (baseline 대비 {gap:+.4f})" if gap is not None else ""
        lines.append(f"  ** best_combo: {{{', '.join(f'{k}={best_combo.get(k)}' for k in varied_keys)}}} "
                      f"-> val_macro_f1={best_combo.get('val_macro_f1'):.4f}{gap_str} **")
    else:
        lines.append("  (best_combo 없음 - 그룹별 최선값이 이미 baseline/개별 변인과 동일했음)")

    return lines


summary_lines = []
summary_lines.append("=== 3주차 파라미터 튜닝 결과 요약 ===")
summary_lines.append("")
summary_lines.extend(summarize_model(
    "SBERT(ko-sroberta)", sbert_results,
    ["max_length", "pooling", "lr", "head"],
))
summary_lines.append("")
summary_lines.extend(summarize_model(
    "PLM(klue-roberta)", plm_results,
    ["max_length", "lr"],
))
summary_lines.append("")

for line in summary_lines:
    print(line)

In [ ]:
summary_lines.append("[방법론 한계 명시]")
summary_lines.append("- dev set을 별도로 분리하지 않음: val로 튜닝하고 val로 보고 - 이 튜닝 결과는")
summary_lines.append("  탐색적 관찰로만 취급하고, 대표 성능은 baseline(3주차_실험계획.md 원칙)을 정본으로 둔다.")
summary_lines.append("- 각 그룹은 baseline에서 변인 하나씩만 바꿔 비교(변인 통제), 전수 조합 탐색이 아님.")
summary_lines.append("")

summary_lines.append("[통합 비교표]")
summary_lines.append(comparison_df.to_string(index=False))

summary_text = "\n".join(summary_lines)

summary_path = ct.TUNING_RESULTS_DIR / "tuning_summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_text)
print(f"저장 완료 -> {summary_path}")

csv_path = ct.TUNING_RESULTS_DIR / "tuning_comparison_all.csv"
comparison_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"저장 완료 -> {csv_path}")